<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Lecture_10/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа № 10: Агенты и мультиагентные системы в психологии

## Введение

В лекции №10 мы познакомились с агентными и мультиагентными системами, которые представляют собой следующий уровень развития ИИ-инструментов для психологии. В отличие от простых чат-ботов, агенты способны:

- **планировать** последовательность действий,
- **использовать инструменты** (поиск по DSM‑5, оценка риска, обращение к базам знаний),
- **запоминать** историю диалога (краткосрочная и долгосрочная память),
- **адаптироваться** к ответам пациента,
- **координировать работу** нескольких специализированных агентов в мультиагентных системах.

Мы разобрали, как **LangChain** помогает создавать агентов с памятью и инструментами, а **LangGraph** позволяет строить мультиагентные системы с состоянием, где агенты обмениваются информацией и совместно решают сложные диагностические задачи. Также мы обсудили долгосрочных персонализированных помощников, тренды 2026 года и этические аспекты применения агентов в клинической практике.

**Цель данной работы** — закрепить практические навыки:

- установки и настройки **Ollama** в Google Colab,
- создания **агентов** с использованием **LangChain** (инструменты, память),
- построения **мультиагентных систем** с **LangGraph**,
- интеграции **RAG** (векторный поиск с Chroma) в агентные системы,
- сравнения single‑agent и multi‑agent подходов,
- критической оценки этических рисков и разработки протоколов безопасного использования.

---

## Подготовка рабочей среды

Все задания выполняются в **Google Colab** (бесплатная облачная среда с GPU). Вам потребуется аккаунт Google и доступ к интернету.

**Минимальные требования:**

- Браузер (Chrome, Firefox, Edge).
- Аккаунт Google для доступа к Colab.
- Около 5–7 ГБ свободного места в Google Drive (для сохранения моделей, опционально).

**Важно:** все примеры кода адаптированы для Colab и используют модель `llama3.1:8b` (около 4.7 ГБ) или `llama3.2:3b` (более лёгкая). Если у вас медленный интернет, можно заменить на `llama3.2:1b`.

Перед началом работы создайте новый ноутбук в Colab и переименуйте его в `Agents_Psychology`.

---

## Часть 1. Теоретические вопросы (для самопроверки)

Письменно ответьте на следующие вопросы (кратко, но содержательно). Это поможет убедиться, что вы понимаете ключевые концепции.

1. Что такое агент в контексте LLM? В чём его отличие от обычного чат-бота? Приведите аналогию из психотерапии.

2. Назовите четыре основных компонента агента. Как каждый из них соотносится с работой психолога?

3. Что такое краткосрочная и долгосрочная память агента? Приведите пример использования в психологическом консультировании.

4. Как LangChain упрощает создание агентов? Какие компоненты LangChain используются для инструментов, памяти и планирования?

5. Что такое LangGraph и для чего он нужен? Чем он отличается от простой цепочки (chain) в LangChain?

6. Опишите архитектуру мультиагентной системы с четырьмя агентами: интервьюер, аналитик, библиотекарь (RAG), супервизор. Как они кооперируются?

7. Что такое Human‑in‑the‑Loop (HITL) и почему он критически важен в клинических агентных системах?

8. Назовите три тренда 2026 года в области долгосрочных персонализированных ИИ-помощников для психического здоровья.

9. Какие этические риски связаны с использованием агентов, которые самостоятельно вызывают инструменты (например, оценку суицидального риска)?

10. **Рефлексивный вопрос:** как вы видите баланс между автономностью агента и человеческим контролем в диагностике психических расстройств? Где должна проходить граница?





## Часть 2. Практические задания на Python

Все задания выполняйте в одном Jupyter Notebook в Colab. Код должен быть снабжён комментариями на русском языке. В конце каждого задания приводите краткий анализ результатов.


### Задание 1. Установка и настройка Ollama в Colab

**Описание.** В этом задании вы подготовите окружение: установите Ollama, запустите сервер и скачаете модель.

**Требуется:**

1. Выполните код из лекции (раздел 7.2 «Подготовка окружения»), который:
   - удаляет старые файлы Ollama,
   - скачивает и устанавливает Ollama с GitHub,
   - запускает сервер с проверкой доступности,
   - скачивает модель `llama3.1:8b` (или `llama3.2:3b`).

2. Проверьте, что модель загружена, выполнив `ollama list`.

3. Отправьте тестовый запрос через `ollama run <model> "Привет"` и получите ответ.

**Что сдать:** скриншоты выполнения всех шагов (терминал в Colab) и краткое описание (1 страница).


In [ ]:
# Ваш код (вставьте сюда код из лекции, адаптированный под задание):

### Задание 2. Создание агента-интервьюера с одним инструментом

**Описание.** В этом задании вы создадите простого агента с одним инструментом — поиском критериев DSM‑5. Агент будет задавать уточняющие вопросы и при необходимости вызывать инструмент.

**Требуется:**

1. Загрузите LLM через `ChatOllama` (используйте модель, скачанную в Задании 1).

2. Создайте инструмент `search_dsm5(query)` (как в лекции), который возвращает диагностические критерии депрессии (можно упрощённо).

3. Настройте память (`ConversationBufferMemory`).

4. Создайте агента с помощью `initialize_agent` с типом `AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION`.

5. Проведите диалог: задайте два вопроса от имени пациента (например, «Мне грустно, я плохо сплю» и «Какие симптомы депрессии?»). Выведите ход рассуждений агента (`verbose=True`).

6. Проанализируйте: вызвал ли агент инструмент, когда это было нужно?



In [ ]:
# Ваш код решения задачи:

### Задание 3. Добавление памяти и второго инструмента

**Описание.** Расширьте агента из Задания 2: добавьте инструмент для оценки суицидального риска и настройте память, чтобы агент запоминал предыдущие ответы.

**Требуется:**

1. Создайте второй инструмент `assess_risk(text)`, который по ключевым словам определяет уровень риска (высокий/умеренный/низкий).

2. Добавьте его в список инструментов агента.

3. Модифицируйте системный промпт, чтобы агент при подозрении на риск обязательно вызывал `assess_risk`.

4. Проведите диалог из трёх сообщений:
   - Пациент: «Мне очень тяжело, ничего не радует».
   - Агент: задаёт уточняющий вопрос.
   - Пациент: «Иногда я думаю, что лучше бы меня не было».
   - (Агент должен вызвать `assess_risk` и отреагировать.)

5. Выведите историю диалога из памяти агента.

**Что сдать:** код, вывод агента, анализ того, как память повлияла на диалог.


In [ ]:
# Ваш код решения задачи:

### Задание 4. Агент-аналитик дневниковых записей

**Описание.** Создайте агента, который анализирует дневниковые записи пациентов: выявляет когнитивные искажения, оценивает эмоциональный тон и предлагает техники КПТ. Используйте код из лекции (раздел 7.4) как основу.

**Требуется:**

1. Определите три инструмента:
   - `detect_cognitive_distortions` — по ключевым словам определяет искажения.
   - `suggest_cbt_technique` — рекомендует технику КПТ.
   - `assess_emotional_tone` — оценивает тон (позитивный/негативный/нейтральный).

2. Создайте агента с памятью (можно без памяти, так как записи независимы).

3. Протестируйте на трёх записях:
   - «Я всегда всё делаю неправильно. Это ужасно!»
   - «Сегодня был хороший день, я гулял с друзьями.»
   - «Я должен был подготовиться лучше, иначе всё провалится.»

4. Для каждой записи выведите анализ агента.

5. Сравните ответы на разные записи: как агент различает когнитивные искажения и эмоциональный тон?


In [ ]:
# Ваш код решения задачи:

### Задание 5. Мультиагентная система с LangGraph (3 агента)

**Описание.** Постройте мультиагентную систему для диагностики депрессии, используя LangGraph. Реализуйте трёх агентов: **Интервьюер**, **Аналитик**, **Супервизор** (без библиотекаря с RAG — это будет в Задании 6).

**Требуется:**

1. Определите состояние `AssessmentState` с полями: `patient_text`, `findings` (список), `risk_level`, `recommendation`.

2. Реализуйте агента-интервьюера: извлекает симптомы из текста пациента (по ключевым словам) и добавляет их в `findings`.

3. Агента-аналитика: на основе `findings` делает предварительный вывод (например, «соответствует депрессивному эпизоду» или «требуется уточнение»).

4. Агента-супервизора: проверяет риски (по наличию суицидальных слов) и выдаёт итоговую рекомендацию.

5. Постройте граф: Интервьюер → Аналитик → Супервизор.

6. Протестируйте на двух текстах:
   - «Мне грустно, я не сплю, ничего не хочется. Иногда думаю о смерти.»
   - «Я тревожусь из-за работы, но в целом настроение нормальное.»

**Что сдать:** код, вывод каждого агента и итоговый отчёт.



In [ ]:
# Ваш код решения задачи:

### Задание 6. Интеграция настоящего RAG в мультиагентную систему

**Описание.** Добавьте в мультиагентную систему агента-библиотекаря, который использует **настоящий RAG** с векторной базой Chroma и эмбеддингами. Используйте код из лекции (раздел 7.6) для создания базы знаний и функции поиска.

**Требуется:**

1. Создайте векторную базу знаний из 5–7 психологических текстов (можно взять из лекции или свои).

2. Реализуйте функцию `search_knowledge(query, top_k=2)`, которая возвращает релевантные фрагменты с источниками.

3. Добавьте агента-библиотекаря в граф между аналитиком и супервизором. Библиотекарь должен:
   - Использовать `search_knowledge` на основе текста пациента.
   - Добавлять найденные фрагменты в `findings`.

4. Обновите супервизора, чтобы он учитывал информацию из RAG.

5. Протестируйте систему на вопросе, который требует обращения к базе (например, «Каковы критерии депрессии по DSM-5?» или «Что такое КПТ?»).

**Что сдать:** код всей системы с RAG, пример запроса и ответа, сравнение с системой без RAG.



In [ ]:
# Ваш код решения задачи:

### Задание 7. Сравнительный анализ Single Agent vs Multi-Agent System

**Описание.** На основе выполненной работы напишите аналитический отчёт (2–3 страницы), в котором сравните:

- **Single Agent** (Задания 2–4) и **Multi-Agent System** (Задания 5–6).
- Какие задачи каждый подход решает лучше?
- В чём преимущества мультиагентной системы с RAG?
- Какие сложности возникли при реализации?
- Какой подход вы бы рекомендовали для использования в клинической практике и почему?

**Что сдать:** отчёт в текстовой ячейке Notebook или отдельным файлом.





```
# Ваш отчёт (текст):
```



## Часть 3. Этический анализ

**Задание 8. Разработка этического протокола для агентных систем в психологии**

**Описание.** Представьте, что вы — руководитель психологического центра. Вы хотите внедрить мультиагентную систему с RAG для поддержки диагностики и первичного скрининга. Напишите этический протокол (2–3 страницы), в котором осветите:

1. **Информированное согласие** — что пациент должен знать об агентах? Какие пункты обязательно включить (автономность агентов, вызов инструментов, сбор данных)?

2. **Human‑in‑the‑Loop** — как организовать контроль? В каких случаях агент должен передавать управление человеку? Кто отвечает за финальный диагноз?

3. **Конфиденциальность и безопасность** — где хранятся данные? Как обеспечивается анонимизация? Как логируются действия агентов?

4. **Обработка ошибок** — что делать при ложных срабатываниях (например, агент ошибочно оценил риск)? Как минимизировать галлюцинации?

5. **Прозрачность** — как объяснить пациенту и коллегам, почему агент принял то или иное решение? Какие записи должны вестись для аудита?

6. **Обучение персонала** — как подготовить психологов к работе с агентной системой? Какие навыки нужны?

**Что сдать:** этический протокол в формате PDF или текстовой ячейкой в Notebook.







```
# Ваш отчёт (текст):
```



## Часть 4. Дополнительное задание (повышенной сложности — по желанию)

**Задание 9. Создание Streamlit‑приложения для агента-интервьюера**

**Описание.** Разработайте простое веб-приложение на Streamlit, которое позволяет пользователю общаться с агентом-интервьюером (из Задания 3) через веб-интерфейс.

**Требуется:**

1. Установите Streamlit: `!pip install streamlit` (в Colab можно использовать `streamlit run` через `ngrok` или `colab-ssh`).

2. Создайте приложение с:
   - Полем ввода текста (сообщение пациента).
   - Кнопкой «Отправить».
   - Отображением истории диалога.
   - Индикацией, какой инструмент был вызван.

3. Запустите приложение и продемонстрируйте его работу (скриншоты).

**Что сдать:** код `app.py`, скриншоты работы.



In [ ]:
# Ваш код для Streamlit-приложения:


## Критерии оценки

| Компонент | Процент | Описание |
|-----------|---------|----------|
| Теоретические вопросы (Часть 1) | 15% | Полнота и правильность ответов |
| Задание 1 (установка Ollama) | 5% | Корректность установки, скриншоты |
| Задание 2 (агент с одним инструментом) | 10% | Работающий агент, вызов инструмента |
| Задание 3 (память и второй инструмент) | 10% | Работа памяти, вызов `assess_risk` |
| Задание 4 (агент-аналитик) | 10% | Корректный анализ трёх записей |
| Задание 5 (мультиагентная система) | 15% | Граф с тремя агентами, рабочие переходы |
| Задание 6 (RAG в мультиагентной системе) | 15% | Интеграция Chroma, поиск и использование |
| Задание 7 (сравнительный анализ) | 5% | Глубина сравнения, аргументированность |
| Задание 8 (этический протокол) | 15% | Полнота, практичность, учёт требований |
| Задание 9 (Streamlit, бонус) | +5% | Работающее приложение, демонстрация |

---

## Требования к сдаче

- Пришлите **один файл** (Jupyter Notebook `.ipynb`) со всеми заданиями, кодом и текстовыми комментариями.
- Для заданий 7 и 8 (отчёт и протокол) используйте текстовые ячейки в Notebook или приложите отдельные файлы (PDF/DOCX).
- Скриншоты вставьте в Notebook.
- Убедитесь, что код выполняется без ошибок (указаны версии библиотек, если требуется).
- Все результаты анализа должны сопровождаться интерпретацией с точки зрения психолога.

---

## Заключение

Данная практическая работа проведёт вас через полный цикл создания агентных систем для психологии — от установки Ollama до построения мультиагентной диагностики с настоящим RAG. Вы не только освоите технические инструменты (LangChain, LangGraph, Chroma), но и научитесь критически оценивать этические аспекты автономных ИИ-систем в клинической практике.

**Главный вывод:** агенты — это мощный инструмент, расширяющий возможности психолога, но их применение требует строгого контроля, прозрачности и ответственности. Будущее — за гибридными системами, где ИИ помогает, но человек принимает финальные решения.

---

**Срок выполнения: 2 недели.**